# Exploring JSON Schemas

## Introduction

In practice you will often encounter a JSON file with little or no documentation — an API response you haven't seen before, a data dump from a colleague, or a third-party export. This notebook covers a systematic technique for mapping out an unknown JSON schema and converting the relevant subset to a pandas DataFrame.

## Objectives

You will be able to:

- Systematically explore an unknown JSON schema using type-checking loops
- Read a JSON schema diagram and relate it to the actual data structure
- Extract the relevant data slice from a deeply nested JSON object
- Expand nested columns in a DataFrame using `map` and `lambda`

---

## The Schema Exploration Technique

When you encounter an unknown JSON file, work top-down:

1. Check the root type — usually `dict` or `list`
2. If `dict`, list the keys and check each value's type
3. Follow the branch that leads toward records (usually a `list` of `dict` items)
4. Repeat until you reach the data you want

A `for` loop over `.keys()` is faster than checking each key manually.

In [ ]:
import json
import pandas as pd

with open('data/exploring_and_transforming_json_schemas/output.json') as f:
    data = json.load(f)

# Step 1: root type
print('root:', type(data))

In [ ]:
# Step 2: keys at root level
data.keys()

In [ ]:
# Step 3: one level down — 'albums' is another dict
print(type(data['albums']))

# Step 4: loop over its keys to see all types at once
for key in data['albums'].keys():
    print(key, '->', type(data['albums'][key]))

At this point the schema looks like this:

<img src="assets/exploring_and_transforming_json_schemas/json_diagram1.JPG" width="550">

Adding what we know about each value type:

<img src="assets/exploring_and_transforming_json_schemas/json_diagram2.JPG" width="550">

The `items` key holds a list — that's almost certainly where the records are.

In [ ]:
items = data['albums']['items']
print(type(items), '->', len(items), 'records')
print('Each item is a:', type(items[0]))
items[0].keys()

---

## Converting to a DataFrame

Pass the list of record dicts directly to `pd.DataFrame()`. Columns are created from the keys.

In [ ]:
df = pd.DataFrame(items)
df.head()

In [ ]:
# Some columns still contain nested data — inspect artists
df.artists.iloc[0]

---

## Expanding Nested Columns

The `artists` column contains a list of dicts. The general strategy: loop over the nested dict's keys and `map` a lambda to extract each one into its own column.

In [ ]:
# Get the keys from the first artist dict
artist_keys = df.artists.iloc[0][0].keys()

new_cols = []
for key in artist_keys:
    col_name = f'artist_{key}'
    # Each artists entry is a list — take the first artist ([0])
    df[col_name] = df.artists.map(lambda x: x[0][key])
    new_cols.append(col_name)

df[new_cols].head()

This is the general pattern for any nested column: `df[col].map(lambda x: x[key])`.

---

## Practice

Explore `data/exploring_and_transforming_json_schemas_lab/disease_data.json` — a health dataset you haven't seen before.

In [ ]:
# Load the file


In [ ]:
# Step 1: check root type and keys


In [ ]:
# Step 2: explore one level down — type-check each key with a for loop


In [ ]:
# Step 3: find the records list and inspect one entry


In [ ]:
# Step 4: find the column names (they're stored in the meta section)
# The DataFrame should have 42 columns


In [ ]:
# Build the DataFrame with proper column names


In [ ]:
# Level-up: create a bar chart of states with the highest asthma rates for adults age 18+
# Hint: filter rows where the 'Topic' column contains 'Asthma' and age group is '18+'


---

## Summary

In this notebook you learned:

- A systematic top-down technique for exploring any unknown JSON: check root type → loop over keys checking types → follow the `list` branch → inspect one item
- `pd.DataFrame(list_of_dicts)` converts a records list directly to a DataFrame
- Nested columns can be expanded with `df[col].map(lambda x: x[key])` — repeat for each nested key
- Column names are often in a `meta` section; extract them before building the DataFrame

Next: [03 — REST APIs and JSON Responses](03_rest_apis_and_json_responses.ipynb)